# 04 — FastAPI Application

Wraps the compiled LangGraph pipeline in a FastAPI app with a single
`POST /ask` endpoint: accepts `AskRequest {"query": str}`, returns the
validated `AskResponse` (`answer`, `sources`, `confidence`).

Writes `main.py` — this is the file `uvicorn` and the `Dockerfile` actually run,
since a `.ipynb` cannot be the target of a `CMD` in Docker.


In [1]:
%%writefile main.py
"""
FastAPI wrapper around the LangGraph Zepto policy-assistant pipeline.

Run locally:
    uvicorn main:app --host 0.0.0.0 --port 7860

MOCK_LLM defaults to mock mode (graded baseline) unless explicitly set to "0"
before the process starts.
"""
from fastapi import FastAPI, HTTPException

from schemas import AskRequest, AskResponse
from graph import ask as run_graph

app = FastAPI(
    title="Zepto Policy Support Assistant",
    description="RAG-backed support assistant over Zepto's delivery, returns, "
                 "membership, tracking, cancellation, gift card, and support policies.",
    version="1.0.0",
)


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/ask", response_model=AskResponse)
def ask_endpoint(request: AskRequest) -> AskResponse:
    try:
        return run_graph(request.query)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


Writing main.py


In [2]:
# Quick in-process check that the app object is importable and wired correctly
# (does not start a server here — see 05_demo.ipynb for live uvicorn calls).
from main import app
print([r.path for r in app.routes])


c:\Users\Srivatsav\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['/openapi.json', '/docs', '/docs/oauth2-redirect', '/redoc', '/health', '/ask']
